# Shot-by-Shot AD Analysis — Character Bank Edition

Supports **Qwen (Unsloth bnb-4bit)** and **Gemini (API)** backends.
Detects faces across shots, clusters them into a character bank,
and injects character names into vision-language prompts.

## Requirements
- GPU: T4 x2 for Qwen (enable in Runtime settings)
- Gemini API key (Kaggle Secret `GEMINI_API_KEY`) for Gemini backend
- Internet: Enable for model download (Qwen) or API calls (Gemini)

## Workflow
1. Run Cell 1 (install) → Runtime → Restart Session
2. Run from Cell 2 onward in order
3. Character bank is detected and injected into Stage 1 & 2 prompts

## Output Files
- `shot_by_shot_output.csv` - Main merged output
- `stage1_descriptions.csv` - Detailed VLM descriptions
- `stage2_audio_descriptions.csv` - Concise AD sentences
- `character_bank.csv` - Per-shot character assignments

In [ ]:
# Install dependencies
# Run this once, then Runtime → Restart Session, then start from Cell 2.
!pip install -q "numpy==1.26.4" --force-reinstall --no-deps
!pip install -U "transformers>=4.51.0" "accelerate>=1.0" "huggingface_hub>=1.22"
!pip install -U "bitsandbytes>=0.46.1" --no-cache-dir
!pip install "unsloth" "unsloth_zoo"
!pip install "scenedetect" "opencv-python-headless>=4.10"
!pip install "google-generativeai" "scikit-learn"
print("\nDONE. Restart session via Runtime menu, then run from Cell 2.")

In [ ]:
# Import libraries
import os, json

# Retrieve secrets from Kaggle
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")

import torch
import pandas as pd
import numpy as np
import cv2
from PIL import Image
from scenedetect import detect, AdaptiveDetector

# Character bank imports
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity

# Check GPU
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.device_count()} device(s)")
    for i in range(torch.cuda.device_count()):
        print(f"  Device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU detected. Gemini backend works without GPU; Qwen needs GPU.")

In [ ]:
# ============================================
# CONFIGURATION - Edit these settings
# ============================================

# --- LLM BACKEND ---
# Choose: "qwen" (local Unsloth bnb-4bit) or "gemini" (API)
LLM_BACKEND = "qwen"

# --- MODEL PATH (Qwen only) ---
MODEL_ID = "unsloth/Qwen2.5-VL-7B-Instruct-unsloth-bnb-4bit"

# --- Gemini model (Gemini only) ---
GEMINI_MODEL = "gemini-2.5-flash"

# Video path
VIDEO_PATH = "/kaggle/input/datasets/yoofun/whitesummer"

# Whisper settings
USE_WHISPER = False
WHISPER_MODEL = "medium"
WHISPER_LANGUAGE = None

# Pipeline settings
USE_VLM = True
USE_STAGE2 = True

# Character bank settings
USE_CHARACTER_BANK = True
CHARACTER_SAMPLE_FRAMES = 16
CHARACTER_THRESHOLD = 0.6

# ============================================

In [ ]:
# ============================================
# CHARACTER BANK — Face Detection & Clustering
# ============================================

CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = cv2.CascadeClassifier(CASCADE_PATH)

def detect_faces(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    return [(int(x), int(y), int(w), int(h)) for (x, y, w, h) in faces]

def _compute_histogram(face_roi):
    hsv = cv2.cvtColor(face_roi, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return hist.flatten()

def extract_face_embeddings(video_path, shot, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    start_frame = int(shot["start_time"] * fps)
    end_frame = int(shot["end_time"] * fps)
    total_frames = end_frame - start_frame
    if total_frames <= 0:
        cap.release()
        return []
    indices = np.linspace(start_frame, end_frame - 1, min(num_frames, total_frames), dtype=int)
    embeddings = []
    for fi in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
        ret, frame = cap.read()
        if not ret:
            continue
        faces = detect_faces(frame)
        if not faces:
            continue
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        roi = frame[y:y+h, x:x+w]
        if roi.size == 0:
            continue
        embeddings.append(_compute_histogram(cv2.resize(roi, (100, 100))))
    cap.release()
    return embeddings

def cluster_faces(embeddings, threshold=0.6):
    if not embeddings:
        return []
    X = np.array(embeddings)
    if X.shape[0] == 1:
        return [0]
    sim = cosine_similarity(X)
    dist = 1.0 - sim
    np.fill_diagonal(dist, 0.0)
    clustering = AgglomerativeClustering(
        n_clusters=None, distance_threshold=threshold,
        metric="precomputed", linkage="average"
    )
    return clustering.fit_predict(dist).tolist()

def build_character_bank(video_path, shots, num_frames=16, threshold=0.6):
    """Detect faces, cluster them, return per-shot character IDs + global mapping."""
    all_embeddings = []
    shot_embedding_counts = []
    for shot in shots:
        embs = extract_face_embeddings(video_path, shot, num_frames=num_frames)
        shot_embedding_counts.append(len(embs))
        all_embeddings.extend(embs)
    if not all_embeddings:
        return [], {}
    global_labels = cluster_faces(all_embeddings, threshold=threshold)
    # How many unique characters?
    unique_chars = sorted(set(global_labels))
    char_map = {c: f"Character_{chr(65+c)}" for c in unique_chars}  # A, B, C...
    results = []
    offset = 0
    for shot, count in zip(shots, shot_embedding_counts):
        labels = global_labels[offset:offset+count]
        offset += count
        results.append({
            "shot_id": shot["shot_id"],
            "character_ids": sorted(set(labels)),
            "character_names": [char_map[l] for l in sorted(set(labels))],
            "face_count": count,
        })
    return results, char_map

In [ ]:
# Load Faster-Whisper model
whisper_model = None
if USE_WHISPER:
    from faster_whisper import WhisperModel
    print(f"Loading Whisper model: {WHISPER_MODEL}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    whisper_model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute_type)
    print(f"Whisper loaded on {device} with {compute_type}")
else:
    print("Whisper disabled")

In [ ]:
# Load LLM — Qwen or Gemini
llm_model = None
llm_processor = None

if USE_VLM or USE_STAGE2:
    if LLM_BACKEND == "qwen":
        import bitsandbytes
        print(f"bitsandbytes: {bitsandbytes.__version__}")
        from unsloth import FastVisionModel
        print(f"Loading Qwen (Unsloth): {MODEL_ID}")
        print(f"GPU count: {torch.cuda.device_count()}")
        llm_model, llm_processor = FastVisionModel.from_pretrained(
            model_name=MODEL_ID,
            load_in_4bit=True,
            low_cpu_mem_usage=True,
        )
        llm_model = FastVisionModel.for_inference(llm_model)
        print("Qwen loaded via Unsloth!")
    elif LLM_BACKEND == "gemini":
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        llm_model = genai.GenerativeModel(GEMINI_MODEL)
        llm_processor = None  # Gemini handles its own preprocessing
        print(f"Gemini backend ready: {GEMINI_MODEL}")
    else:
        raise ValueError(f"Unknown LLM_BACKEND: {LLM_BACKEND}")
else:
    print("VLM/Stage2 disabled, skipping LLM load")

In [ ]:
# Shot detection
def detect_shots(video_path, threshold=27.0):
    scene_list = detect(video_path, AdaptiveDetector(adaptive_threshold=threshold))
    shots = []
    for idx, (start, end) in enumerate(scene_list):
        shots.append({
            "shot_id": idx + 1,
            "start_time": start.get_seconds(),
            "end_time": end.get_seconds()
        })
    return shots

shots = detect_shots(VIDEO_PATH)
print(f"Found {len(shots)} shots")
pd.DataFrame(shots)

In [ ]:
# Character bank detection
character_bank = []
char_map = {}
if USE_CHARACTER_BANK:
    print("Detecting character bank...")
    character_bank, char_map = build_character_bank(
        VIDEO_PATH, shots,
        num_frames=CHARACTER_SAMPLE_FRAMES,
        threshold=CHARACTER_THRESHOLD
    )
    # Create lookup: shot_id -> character names
    char_lookup = {c["shot_id"]: c["character_names"] for c in character_bank}
    print(f"Found {len(char_map)} unique character(s): {list(char_map.values())}")
    
    # Save character bank
    char_df = pd.DataFrame(character_bank)
    char_df.to_csv("/kaggle/working/character_bank.csv", index=False)
    print("Character bank saved to: character_bank.csv")
else:
    print("Character bank disabled")
    char_lookup = {}

In [ ]:
# Edit character names — replace auto-generated A/B/C with real names
# The auto-detection assigns Character_A, Character_B, etc.
# Update this mapping with the actual character names from your video.
if USE_CHARACTER_BANK and char_map:
    char_map = {
        -1: "Unknown",
        0:  "Character_A",  # <-- replace with actual name
        1:  "Character_B",  # <-- replace with actual name
        2:  "Character_C",  # <-- replace with actual name
        # Add more entries as needed
    }
    # Rebuild char_lookup with updated names
    char_lookup = {}
    for c in character_bank:
        names = [char_map.get(cid, f"Character_{chr(65+cid)}") for cid in c["character_ids"]]
        char_lookup[c["shot_id"]] = names
    print("Updated character names:", list(set(char_map.values())))
    print(char_lookup)

In [ ]:
# Whisper transcription
def transcribe_video(video_path, model, language=None):
    segments, info = model.transcribe(
        video_path, language=language, beam_size=5, vad_filter=True
    )
    return [{"text": seg.text.strip(), "start_time": seg.start, "end_time": seg.end} for seg in segments]

if USE_WHISPER and whisper_model:
    print("Transcribing video...")
    subtitles = transcribe_video(VIDEO_PATH, whisper_model, WHISPER_LANGUAGE)
    print(f"Found {len(subtitles)} subtitle segments")
else:
    print("Whisper disabled")
    subtitles = []

In [ ]:
# Merge shots and subtitles
def merge_shots_subtitles(shots, subtitles):
    results = []
    for shot in shots:
        overlapping = [s for s in subtitles
                      if s["start_time"] < shot["end_time"] and s["end_time"] > shot["start_time"]]
        subtitle_text = " ".join([s["text"] for s in overlapping]) if overlapping else ""
        results.append({
            "shot_id": shot["shot_id"],
            "start_time": shot["start_time"],
            "end_time": shot["end_time"],
            "subtitle": subtitle_text
        })
    return pd.DataFrame(results)

merged_df = merge_shots_subtitles(shots, subtitles)
merged_df

In [ ]:
# VLM description (Stage 1) — Qwen or Gemini with character bank
import warnings
warnings.filterwarnings("ignore", message=".*Both max_new_tokens and max_length.*")

def extract_frames(video_path, shot, num_frames=8):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    start_frame = int(shot["start_time"] * fps)
    end_frame = int(shot["end_time"] * fps)
    frame_indices = np.linspace(start_frame, end_frame, num_frames, dtype=int)
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames

def build_vlm_prompt(shot_id):
    """Build Stage 1 prompt with character names if available."""
    prompt = "請簡短描述這段影片片段發生了什麼事。請使用繁體中文。專注於角色、動作和環境。"
    if USE_CHARACTER_BANK and shot_id in char_lookup and char_lookup[shot_id]:
        names = "、".join(char_lookup[shot_id])
        prompt += f" 已知角色：{names}。請在描述中使用這些角色名稱。"
    return prompt

def describe_frames_qwen(frames, model, processor, shot_id):
    prompt = build_vlm_prompt(shot_id)
    content = [{"type": "text", "text": prompt}]
    for frame in frames:
        content.append({"type": "image", "image": frame})
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, output_ids)]
    return processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

def describe_frames_gemini(frames, model, shot_id):
    prompt = build_vlm_prompt(shot_id)
    contents = [prompt]
    contents.extend(frames)
    response = model.generate_content(contents)
    return response.text.strip()

# Stage 2 summarizer — Qwen or Gemini
def build_summarize_prompt(desc, word_limit):
    prompt = f"""請將以下描述濃縮成一句簡潔的口述影像句子。使用繁體中文。
專注於角色、動作和關鍵物件。
使用名字或代名詞。避免提到鏡頭。
限制在 {word_limit} 個字以內。

Input: {desc}

Output:"""
    return prompt

def summarize_qwen(desc, model, processor, word_limit):
    prompt = build_summarize_prompt(desc, word_limit)
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, output_ids)]
    result = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    if not result.endswith("."):
        result += "."
    return result

def summarize_gemini(desc, model, word_limit):
    prompt = build_summarize_prompt(desc, word_limit)
    response = model.generate_content(prompt)
    result = response.text.strip()
    if not result.endswith("."):
        result += "."
    return result

# Run Stage 1
descriptions_dict = {}
if USE_VLM and llm_model:
    print(f"Running Stage 1: VLM descriptions ({LLM_BACKEND})...")
    for shot in shots:
        shot_id = shot["shot_id"]
        try:
            frames = extract_frames(VIDEO_PATH, shot)
            if LLM_BACKEND == "qwen":
                desc = describe_frames_qwen(frames, llm_model, llm_processor, shot_id)
            else:
                desc = describe_frames_gemini(frames, llm_model, shot_id)
            descriptions_dict[shot_id] = desc
            print(f"Shot {shot_id}: OK")
        except Exception as e:
            print(f"Shot {shot_id}: Failed - {e}")
            descriptions_dict[shot_id] = ""
    merged_df["video_description"] = merged_df["shot_id"].map(descriptions_dict).fillna("")
    stage1_df = merged_df.copy()
    stage1_df.to_csv("/kaggle/working/stage1_descriptions.csv", index=False)
    print("Stage 1 saved to: stage1_descriptions.csv")
else:
    print("VLM disabled")

# Run Stage 2
if USE_STAGE2 and llm_model and descriptions_dict:
    print(f"\nRunning Stage 2: Summarizing to AD ({LLM_BACKEND})...")
    ad_sentences = []
    for _, row in merged_df.iterrows():
        duration = row["end_time"] - row["start_time"]
        word_limit = max(1, int(duration * 3))
        try:
            if LLM_BACKEND == "qwen":
                ad = summarize_qwen(row["video_description"], llm_model, llm_processor, word_limit)
            else:
                ad = summarize_gemini(row["video_description"], llm_model, word_limit)
            ad_sentences.append(ad)
        except:
            ad_sentences.append("")
    merged_df["ad_sentence"] = ad_sentences
    stage2_df = merged_df[["shot_id", "start_time", "end_time", "ad_sentence"]].copy()
    stage2_df.to_csv("/kaggle/working/stage2_audio_descriptions.csv", index=False)
    print("Stage 2 saved to: stage2_audio_descriptions.csv")
else:
    print("Stage 2 disabled")

merged_df

In [ ]:
# Save outputs
output_path = "/kaggle/working/shot_by_shot_output.csv"
merged_df.to_csv(output_path, index=False)
print(f"Main output saved to: {output_path}")
print("\nOutput files:")
print("  - shot_by_shot_output.csv")
if USE_VLM and llm_model:
    print("  - stage1_descriptions.csv")
if USE_STAGE2 and llm_model:
    print("  - stage2_audio_descriptions.csv")
if USE_CHARACTER_BANK:
    print("  - character_bank.csv")

In [ ]:
# Download links
from IPython.display import FileLink, display
print("Download links:")
display(FileLink("/kaggle/working/shot_by_shot_output.csv"))
if USE_VLM and llm_model:
    display(FileLink("/kaggle/working/stage1_descriptions.csv"))
if USE_STAGE2 and llm_model:
    display(FileLink("/kaggle/working/stage2_audio_descriptions.csv"))
if USE_CHARACTER_BANK:
    display(FileLink("/kaggle/working/character_bank.csv"))